# Phase 1 — Data Exploration

Validate the DREAM4 data pipeline and understand the dataset structure
before running any inference.

Checks:
- All 5 networks × 2 sizes load correctly
- Gold standard shapes and sparsity
- Timeseries shapes and perturbation structure
- Knockout shapes and wildtype baseline

In [ ]:
import pathlib, sys
ROOT = pathlib.Path('../../../../')
sys.path.insert(0, str(ROOT / 'src'))

import numpy as np
import matplotlib.pyplot as plt

from grn_world_model.data_loader import load_all_networks, load_dream4_network

DATA_DIR = ROOT / 'data'

## 1. Network Summary Table

In [ ]:
import pandas as pd

rows = []
for size in [10, 100]:
    for net in load_all_networks(DATA_DIR, size=size):
        gs = net.gold_standard
        rows.append({
            'Network': f'net{net.network_id} (Size{size})',
            'N_genes': net.n_genes,
            'True edges': int(gs.sum()) if gs is not None else 'N/A',
            'Sparsity': f"{gs.sum() / (net.n_genes**2 - net.n_genes):.3f}" if gs is not None else 'N/A',
            'N_timeseries': len(net.timeseries),
            'N_knockouts': len(net.knockout_pairs),
            'Gold standard': 'yes' if gs is not None else 'missing',
        })

df = pd.DataFrame(rows)
print(df.to_string(index=False))

## 2. Timeseries Structure (Size10, all 5 networks)

In [ ]:
nets10 = load_all_networks(DATA_DIR, size=10)
fig, axes = plt.subplots(5, 5, figsize=(16, 12), sharey=False)

for row, net in enumerate(nets10):
    for col, ts in enumerate(net.timeseries):
        ax = axes[row][col]
        for g in range(net.n_genes):
            ax.plot(ts.timepoints, ts.expression[:, g], lw=0.8, alpha=0.7)
        ax.axvline(500, color='k', lw=0.6, ls='--', alpha=0.4)
        if col == 0:
            ax.set_ylabel(f'net{net.network_id}', fontsize=8)
        if row == 0:
            ax.set_title(f'Series {col+1}', fontsize=8)
        ax.tick_params(labelsize=6)

fig.suptitle('All Size10 timeseries (rows=networks, cols=series, dashed=t=500)', fontsize=11)
plt.tight_layout()
plt.savefig(DATA_DIR.parent / 'results' / 'grn_world_model' / 'phase1_data' / 'all_timeseries.png',
            dpi=120, bbox_inches='tight')
plt.show()

## 3. Gold Standard Adjacency Matrices

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(14, 3))
for ax, net in zip(axes, nets10):
    ax.imshow(net.gold_standard, cmap='Blues', vmin=0, vmax=1)
    ax.set_title(f'net{net.network_id}\n{int(net.gold_standard.sum())} edges', fontsize=9)
    ax.set_xlabel('Target'); ax.set_ylabel('Source')
    ax.set_xticks([]); ax.set_yticks([])

fig.suptitle('Gold standard adjacency matrices (Size10)', fontsize=11)
plt.tight_layout()
plt.show()

## 4. ODE Solver Sanity Check

In [ ]:
from grn_world_model.ode_model import validate_ode

for net in nets10:
    validate_ode(net)
    print()

## 5. Perturbation Vectors

In [ ]:
net = nets10[0]
fig, axes = plt.subplots(1, 5, figsize=(14, 2.5), sharey=True)
for ax, ts in zip(axes, net.timeseries):
    bars = ax.bar(range(net.n_genes), ts.perturbation,
                  color=['C0' if v > 0 else 'C3' for v in ts.perturbation])
    ax.axhline(0, color='k', lw=0.6)
    ax.set_xticks(range(net.n_genes))
    ax.set_xticklabels([f'G{i+1}' for i in range(net.n_genes)], rotation=90, fontsize=7)
    ax.set_title(f'Series {net.timeseries.index(ts)+1}', fontsize=9)

axes[0].set_ylabel('x(t=0) − wildtype')
fig.suptitle(f'Perturbation vectors net1 (approx: x(t=0) − wildtype)', fontsize=11)
plt.tight_layout()
plt.show()
print('Note: DREAM4 timeseries are multi-gene perturbations. '
      'The perturbation vector is the approximate basal shift used in the ODE.')